# 05 · Supplier Scorecard
Turn the quote-level data into one row per supplier scored 0–100 on five **configurable** weighted dimensions, then assign a category and a recommended action.

In [ ]:
%matplotlib inline
import sys, pathlib
ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40, "display.width", 160)
from src import config as C
print("project root:", ROOT)

In [ ]:
from src import supplier_scoring as ss
df = pd.read_csv(C.PROCESSED_QUOTES_CSV)
print('weights:', C.SCORECARD_WEIGHTS)
sc = ss.score_suppliers(df)
sc[['supplier_name','country','final_supplier_score','supplier_category','recommended_action']].head(10)

## Category & action distribution

In [ ]:
display(sc.supplier_category.value_counts().to_frame('suppliers'))
display(sc.recommended_action.value_counts().to_frame('suppliers'))

## Scorecard profile by category

In [ ]:
dims = {'cost_competitiveness_score':'Cost','delivery_reliability_score':'Delivery',
        'quality_performance_score':'Quality','risk_exposure_score':'Risk','sustainability_score_norm':'Sustainability'}
order=[c for c in ['Strategic Partner','Reliable but Expensive','Cost Efficient','High Risk','Needs Improvement'] if c in sc.supplier_category.unique()]
mat = sc.groupby('supplier_category')[list(dims)].mean().reindex(order).rename(columns=dims)
fig, ax = plt.subplots(figsize=(10,5.5))
sns.heatmap(mat, annot=True, fmt='.0f', cmap='RdYlGn', vmin=0, vmax=100, linewidths=.5, ax=ax)
ax.set_title('Average scorecard profile by category'); ax.set_ylabel(''); plt.show()

## Validation — does the scorecard rediscover the latent archetypes?
The generating archetype is hidden ground truth (it would not exist in real data).

In [ ]:
master = pd.read_csv(C.SUPPLIER_MASTER_CSV)[['supplier_id','archetype']]
j = sc.merge(master, on='supplier_id')
pd.crosstab(j.archetype, j.supplier_category)

## Top and bottom suppliers

In [ ]:
cols=['supplier_name','country','final_supplier_score','supplier_category','recommended_action']
print('TOP 5'); display(sc.nlargest(5,'final_supplier_score')[cols])
print('BOTTOM 5'); display(sc.nsmallest(5,'final_supplier_score')[cols])

**Business decision.** Prioritize the strategic partners, renegotiate with the premium suppliers that hold the largest spend share, and reduce dependency on the high-risk, high-spend vendors concentrated in a single region.